# **PPO ON RACE CAR ENV**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import numpy as np

import gymnasium as gym


### **ENV SETUP**

In [ ]:
env = gym.make("Ant-v5", render_mode = 'human')

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
max_action = env.action_space.high[0]
min_action = env.action_space.low[0]

print(f'state dim: {state_dim} | action dim: {action_dim} | max action: {max_action} | min action: {min_action}')


state dim: 105 | action dim: 8 | max action: 1.0 | min action: -1.0


### **DEVICE**

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


### **HELPER FUNCTION**

In [ ]:
def safe_tensor(x):
    
    return x if torch.is_tensor(x) else torch.tensor(x, dtype = torch.float32).to(device)


### **ASSEMBLY**

In [ ]:
head_1 = 128
head_2 = 256
head_3 = 256
head_4 = 128

hidden_size = 128
hidden_size_2 = 256


### **HYPER X**


In [ ]:
class hyper_x(nn.Module):
    
    def __init__(self, input_dim, output_dim, hidden_size = hidden_size, hidden_size_2 = hidden_size_2):
        super(hyper_x, self).__init__()
        
        # fc1 
        
        self.fc1 = nn.Linear(input_dim, hidden_size)
        self.norm1 = nn.LayerNorm(hidden_size)
        
        # fc2
        
        self.fc2 = nn.Linear(hidden_size, hidden_size_2)
        self.norm2 = nn.LayerNorm(hidden_size_2)
        
        # MHA
        
        self.mha = nn.MultiheadAttention(hidden_size_2, num_heads = 2, batch_first = True)
        self.mha_norm = nn.LayerNorm(hidden_size_2)
        
        # fc3
        
        self.fc3 = nn.Linear(hidden_size_2, hidden_size_2)
        self.norm3 = nn.LayerNorm(hidden_size_2)
        
        # fc4
        
        self.fc4 = nn.Linear(hidden_size_2, hidden_size)
        self.norm4 = nn.LayerNorm(hidden_size)
        
        # final output later
        
        self.out = nn.Linear(hidden_size, output_dim)
        self.out_norm = nn.LayerNorm(output_dim)
        
    def forward(self, cat):
        
        # fc
        
        x = F.silu(self.norm1(self.fc1(cat)))
        x = F.silu(self.norm2(self.fc2(x)))
        
        # mha
        
        if x.dim() == 2:
            
            x = x.unsqueeze(1)
            
            mha_out, _ = self.mha(x, x, x, need_weights = False)
            
            mha_out = mha_out.squeeze(1)
            x = x.squeeze(1)
            
        else:
            
            mha_out, _ = self.mha(x, x, x, need_weights = False)
            
        x = self.mha_norm(x + mha_out)
        
        # fc
        
        x = F.silu(self.norm3(self.fc3(x)))
        x = F.silu(self.norm4(self.fc4(x)))
        
        out = F.silu(self.out_norm(self.out(x)))
        
        
        return out        


### **POLICY**

In [ ]:
class policy_net(nn.Module):
    
    def __init__(self, state_dim = state_dim, head_1 = head_1, head_2 = head_2, head_3 = head_3, head_4 = head_4, max_action = max_action):
        super(policy_net, self).__init__()
        
        # hyper
        
        self.hyper = hyper_x(state_dim, head_1)
        
        # policy mlp
        
        self.policy = nn.Sequential(
            
            nn.Linear(head_1, head_2),
            nn.LayerNorm(head_2),
            nn.SiLU(),
            
            nn.Linear(head_2, head_3),
            nn.LayerNorm(head_3),
            nn.SiLU(),
            
            nn.Linear(head_3, head_4),
            nn.LayerNorm(head_4),
            nn.SiLU()
            
        )
        
        # mu and log std heads
        
        self.mu = nn.Linear(head_4, action_dim)
        self.log_std = nn.Linear(head_4, action_dim)
        
        # max action
        
        self.max_action = max_action
        
        # norma;ization
        
        self.apply(self.init_weight)
        
    def forward(self, state, deterministic = False):
        
        # state -> hyper
        
        x = self.hyper(state)
        
        # x -> policy
        
        policy = self.policy(x)
        
        # mu and log head
        
        mu = self.mu(policy)
        
        if deterministic:
            
            return torch.tanh(mu) * self.max_action

        
        log_std = self.log_std(policy)
        log_std = torch.clamp(log_std, -20, 2)
        std = torch.exp(log_std)
        
        # reparameterization trick
        
        dist = torch.distributions.Normal(mu, std)
        z = dist.rsample()
        tanh_z = torch.tanh(z)
        action = tanh_z * self.max_action
        
        # entropy
        
        entropy = dist.entropy().sum(dim = -1, keepdim = True)
        
        # log prob
        
        log_prob = dist.log_prob(z).sum(dim = -1, keepdim = True)
        squash = torch.log(1 - tanh_z.pow(2) + 1e-6).sum(dim = -1, keepdim = True)
        log_prob = log_prob - squash

        return action, log_prob, entropy
    
    
    def get_log_probs(self, states, actions):
        
        # crete dist
        
        x = self.hyper.forward(states)
        
        policy = self.policy(x)
        
        mu = self.mu(policy)
        log_std = self.log_std(policy)
        
        log_std = log_std.clamp(-20, 2)
        std = torch.exp(log_std)
        
        
        dist = torch.distributions.Normal(mu, std)
        
        # create sample
        
        z = 0.5 * torch.log(((1 + actions) / self.max_action) / ((1 - actions) / self.max_action) + 1e-7)
        
        # entropy
        
        entropy = dist.entropy().sum(dim = -1, keepdim = True)
        
        # log probs
        
        log_prob = dist.log_prob(z).sum(dim = -1, keepdim = True)
        squash = torch.log(1 - torch.tanh(z).pow(2) + 1e-6).sum(dim = -1, keepdim = True)
        log_prob = log_prob - squash
        
        return log_prob, entropy
    
    
    def init_weight(self, m):
        
        if isinstance(m, nn.Linear):
            
            nn.init.orthogonal_(m.weight)
            
            if m.bias is not None:
                
                nn.init.zeros_(m.bias)


### **SET UP**

In [ ]:
POLICY_NET = policy_net().to(device)

print(POLICY_NET)


policy_net(
  (hyper): hyper_x(
    (fc1): Linear(in_features=105, out_features=128, bias=True)
    (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (fc2): Linear(in_features=128, out_features=256, bias=True)
    (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (mha): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (mha_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (fc3): Linear(in_features=256, out_features=256, bias=True)
    (norm3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (fc4): Linear(in_features=256, out_features=128, bias=True)
    (norm4): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (out): Linear(in_features=128, out_features=128, bias=True)
    (out_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (policy): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): LayerNor

### **VALUE NETWORK**

In [9]:
class value_net(nn.Module):
    
    def __init__(self, state_dim = state_dim,  head_1 = head_1, head_2 = head_2, head_3 = head_3, head_4 = head_4):
        super(value_net, self).__init__()
        
        # pre prcoess
        
        self.pre_process = nn.Linear(state_dim, head_1)
        self.norm = nn.LayerNorm(head_1)
        
        # post process
        
        self.post_process = nn.Sequential(
            
            nn.Linear(head_1, head_2),
            nn.LayerNorm(head_2),
            nn.SiLU(),
            
            nn.Linear(head_2, head_3),
            nn.LayerNorm(head_3),
            nn.SiLU(),
            
            nn.Linear(head_3, head_4),
            nn.LayerNorm(head_4),
            nn.SiLU()
            
        )
        
        # value head
        
        self.value = nn.Linear(head_4, 1)
    
        # normalization
        
        self.apply(self.init_weight)
        
    def forward(self, state):
        
        # pre process
        
        pre = F.silu(self.norm(self.pre_process(state)))
        
        # post 
        
        post = self.post_process(pre)
        
        # value
        
        value = self.value(post)
        
        return value
        
    def init_weight(self, m):
        
        if isinstance(m, nn.Linear):
            
            nn.init.orthogonal_(m.weight)
            
            if m.bias is not None:
                
                nn.init.zeros_(m.bias)
    

### **SETUP**

In [ ]:
VALUE_NET = value_net().to(device)

print(VALUE_NET)


value_net(
  (pre_process): Linear(in_features=105, out_features=128, bias=True)
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (post_process): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): SiLU()
    (3): Linear(in_features=256, out_features=256, bias=True)
    (4): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (5): SiLU()
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (8): SiLU()
  )
  (value): Linear(in_features=128, out_features=1, bias=True)
)


### **OPTIMIZER**

In [ ]:
# lr

policy_lr = 3e-4
value_lr = 1e-4

T_max = 540

# POLICY

POLICY_OPTIMIZER = optim.AdamW(POLICY_NET.parameters(), policy_lr, weight_decay = 0)

POLICY_SCHEDULER = optim.lr_scheduler.CosineAnnealingLR(POLICY_OPTIMIZER, T_max, eta_min = 1e-5)

# VALUE

VALUE_OPTIMIZER = optim.AdamW(VALUE_NET.parameters(), value_lr, weight_decay = 1e-6)

VALUE_SCHEDULER = optim.lr_scheduler.CosineAnnealingLR(VALUE_OPTIMIZER, T_max, eta_min = 1e-5)


### **BUFFER**

In [ ]:
class roller_buffer:
    
    def __init__(self):
        
        self.buffer = []
        
    def add(self, state, action, log_prob, reward, done, next_state):
        
        # safe tensor
        
        self.buffer.append({
            
            'states': safe_tensor(state),
            'actions': safe_tensor(action),
            'log_probs': safe_tensor(log_prob),
            'rewards': safe_tensor(reward),
            'dones':  safe_tensor(done),
            'next_states': safe_tensor(next_state)
            
        })
        
    def sample(self):
        
        # stack
        
        states = [i['states'] for i in self.buffer]
        actions = [i['actions'] for i in self.buffer]
        log_probs = [i['log_probs'] for i in self.buffer]
        rewards = [i['rewards'] for i in self.buffer]
        dones = [i['dones'] for i in self.buffer]
        next_states = [i['next_states'] for i in self.buffer]
        
        def safe_stack(x):
            
            return torch.stack(x).to(device)
        
        # batch return
        
        batch = {
            
            'states': safe_stack(states),
            'actions': safe_stack(actions),
            'log_probs': safe_stack(log_probs),
            'rewards': safe_stack(rewards),
            'dones': safe_stack(dones),
            'next_states': safe_stack(next_states)
            
        }
        
        return batch
    
    def clear(self):
        
        self.buffer.clear()


### **SET UP**

In [ ]:
BUFFER = roller_buffer()


### **LOSS FUNC**

In [14]:
class loss_func:
    
    def __init__(self, gamma, entropy_coef, gae_lam, value_coef, clip_epsilon, POLICY_NET = POLICY_NET, VALUE_NET = VALUE_NET, BUFFER = BUFFER, VALUE_OPTIMIZER = VALUE_OPTIMIZER, VALUE_SCHEDULER = VALUE_SCHEDULER, POLICY_OPTIMIZER = POLICY_OPTIMIZER, POLICY_SCHEDULER = POLICY_SCHEDULER):
        
        # network
        
        self.policy = POLICY_NET
        self.value = VALUE_NET
        
        # buffer
        
        self.buffer = BUFFER
        
        # optimizer
        
        self.policy_opt = POLICY_OPTIMIZER
        self.policy_sch = POLICY_SCHEDULER
        
        self.value_opt = VALUE_OPTIMIZER
        self.value_sch = VALUE_SCHEDULER
        
        # hyper params
        
        self.gamma = gamma
        self.entropy_coef = entropy_coef
        self.gae_lam = gae_lam
        self.value_coef = value_coef
        self.clip_epsilon = clip_epsilon
        
    def compute_gae(self, rewards, dones, value, last_value):
        
        values = torch.cat([value, last_value], dim = 0)
        
        advantages = []
        gae = 0
        
        for step in reversed(range(len(rewards))):
            
            delta = rewards[step] + self.gamma * (1 - dones[step]) * values[step + 1] - values[step]
            
            gae = delta + self.gamma * self.gae_lam * (1 - dones[step]) * gae
            
            advantages.insert(0, gae)
            
        advantages = safe_tensor(advantages).unsqueeze(-1)
        
        returns = advantages + values[:-1]
        
        returns = safe_tensor(returns)
        
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-7)
        returns = (returns - returns.mean()) / (returns.std() + 1e-7)
        
        return advantages.detach(), returns.detach()
    
    def value_loss(self, states, returns):
        
        value = self.value.forward(states)
        
        v_loss = F.mse_loss(value, returns)
        
        value_loss = v_loss * self.value_coef
        
        return value_loss
    
    def policy_loss(self, old_log_probs, states, actions, advantages):
        
        log_probs, entropy = self.policy.get_log_probs(states, actions)
        
        ratio = torch.exp(log_probs - old_log_probs.detach())
        
        surr1 = ratio * advantages
        
        surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advantages
        
        surrogate_loss = -torch.min(surr1, surr2).mean()
        
        policy_loss = surrogate_loss - self.entropy_coef * entropy.mean()
        
        return policy_loss
    
    
    def update(self, verbose = False):
        
        # sample
        
        batch = self.buffer.sample()
        
        # unpack
        
        states = batch['states']
        actions = batch['actions']
        old_log_probs = batch['log_probs']
        rewards = batch['rewards']
        dones = batch['dones']
        next_states = batch['next_states']
         
        # shape check
        
        dones = dones.view(-1, 1)
        rewards = rewards.view(-1, 1)
        
        if verbose:
            
            print(f'states: {states.shape} | actions: {actions.shape} | old log probs: {old_log_probs.shape} | rewards: {rewards.shape} |  '
                  f' dones: {dones.shape} | next states: {next_states.shape}')
        
        # compute value and last value
        
        value = self.value.forward(states)
        
        if verbose:
        
            print(f'value: {value.shape}')
        
        with torch.no_grad():
            
            last_state = next_states[-1:]
            last_value = self.value.forward(last_state)
            
            if verbose:
                
                print(f'last value: {last_value.shape}')
        
        # compute gae
        
        advantages, returns = self.compute_gae(rewards, dones, value, last_value)
        
        # policy loss
        
        
        policy_loss = self.policy_loss(old_log_probs, states, actions, advantages)
        
        # optimize
        
        self.policy_opt.zero_grad()
        policy_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy.parameters(), max_norm = 0.5)
        self.policy_opt.step()
        self.policy_sch.step()
        
        # value loss
        
        value_loss = self.value_loss(states, returns)
        
        # optimize
        
        self.value_opt.zero_grad()
        value_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.value.parameters(), max_norm = 0.5)
        self.value_opt.step()
        self.value_sch.step()
        
        return policy_loss.item(), value_loss.item()
    

### **SETUP**

In [ ]:
# hyper params

value_coef = 0.5
clip_epsilon = 0.15
entropy_coef = 0.005
gamma = 0.99
gae_lam = 0.95

# setup

LOSS_FUNCTION = loss_func(gamma, entropy_coef, gae_lam, value_coef, clip_epsilon)


### **TRAINING**

In [ ]:
def train_loop(epochs, inner_steps, POLICY_NET = POLICY_NET, BUFFER = BUFFER, LOSS_FUNCTION = LOSS_FUNCTION, env = env):
    
    total_ep_rewards = []
    max_reward = -np.inf
    
    for epoch in range(epochs):
        
        # track reward
        
        total_reward = 0.0
        
        # reset env
        
        obs, _ = env.reset()
        obs = safe_tensor(obs).unsqueeze(0)
        
        # reset buffer
        
        BUFFER.clear()
        
        # collect data
        
        for _ in range(inner_steps):
                        
            action, log_prob, _ = POLICY_NET.forward(obs)
            
            action_np = action.detach().cpu().numpy()[0]
            
            next_obs, reward, termination, timeout, _ = env.step(action_np)
            
            done = termination or timeout
            
            total_reward += reward
            
            BUFFER.add(obs.squeeze(0), action.squeeze(0), log_prob.squeeze(0), reward, done, next_obs)
            
            if done:
                
                break
            
        # cal max reward
        
        if total_reward > max_reward: max_reward = total_reward
        
        # cal loss
        
        policy_loss, value_loss = LOSS_FUNCTION.update()
        
        # logging
        
        print('-' * 100)
        print()
        print(f'Epoch: {epoch}')
        print()
        print(f'Policy Loss: {policy_loss:.3f} | Value loss: {value_loss:.3f}')
        print()
        print(f'max reward: {max_reward:.2f} | Reward: {total_reward:.2f}')
        print()
        print('-' * 100)
        
        total_ep_rewards.append(total_reward)
        
    return total_ep_rewards


In [ ]:
train_loop(540, 1024)


----------------------------------------------------------------------------------------------------

Epoch: 0

Policy Loss: -0.06 | Value loss: 0.57

max reward: -776.72 | Reward: -776.72

----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------

Epoch: 1

Policy Loss: -0.04 | Value loss: 0.51

max reward: -137.11 | Reward: -137.11

----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------

Epoch: 2

Policy Loss: -0.04 | Value loss: 0.50

max reward: -21.19 | Reward: -21.19

----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------

Epoch: 3

Policy Loss: -0.0